# Queues and Deques

```{contents}
:local:
:depth: 2
```


```{index} queue; FIFO
```

## Queue Behavior



A **queue** is an abstract data type where the earliest added item is the next item removed. This access rule is called **first in, first out**, or **FIFO**.

The core operations are:

- `Enqueue(value)` adds a value to the back.
- `Dequeue()` removes and returns the front value.
- `Peek()` returns the front value without removing it.
- `Count` reports how many values are waiting.


In [ ]:
using System;
using System.Collections.Generic;

var serviceLine = new Queue<string>();
serviceLine.Enqueue("Amina");
serviceLine.Enqueue("Diego");
serviceLine.Enqueue("Mina");

Console.WriteLine($"Serving: {serviceLine.Dequeue()}");
Console.WriteLine($"Next: {serviceLine.Peek()}");
Console.WriteLine($"Still waiting: {serviceLine.Count}");


The first customer enqueued is the first customer served. Unlike a list, a queue communicates that callers should not pick arbitrary positions from the middle.


```{index} queue; worklist
```

## Worklists and Scheduling



Queues often represent work that should be processed in arrival order: help-desk tickets, print jobs, messages, or search states. This pattern is called a **worklist** because the collection stores work that remains to be done.


In [ ]:
using System;
using System.Collections.Generic;

var work = new Queue<string>();
work.Enqueue("Validate order");
work.Enqueue("Charge card");
work.Enqueue("Send receipt");

int step = 1;
while (work.Count > 0)
{
    string task = work.Dequeue();
    Console.WriteLine($"Step {step}: {task}");
    step++;
}


A queue makes the scheduling policy explicit: new work joins the back, and existing work leaves from the front.


```{index} ring buffer; queue implementation
```

## Ring Buffers



An array-backed queue cannot simply remove index `0` by shifting every element left. That would make each dequeue expensive. A **ring buffer** solves this by letting the front and back indexes wrap around the array.

For a bounded queue, the important state is:

- `front`: the index of the next item to remove
- `back`: the index where the next item will be added
- `count`: the number of stored items


In [ ]:
using System;

int capacity = 4;
int front = 0;
int back = 0;
int count = 0;

void Enqueue(string value, string[] items)
{
    if (count == items.Length)
    {
        throw new InvalidOperationException("Queue is full.");
    }

    items[back] = value;
    back = (back + 1) % items.Length;
    count++;
}

string Dequeue(string[] items)
{
    if (count == 0)
    {
        throw new InvalidOperationException("Queue is empty.");
    }

    string value = items[front];
    items[front] = null!;
    front = (front + 1) % items.Length;
    count--;
    return value;
}

var items = new string[capacity];
Enqueue("A", items);
Enqueue("B", items);
Enqueue("C", items);
Console.WriteLine(Dequeue(items));
Enqueue("D", items);
Enqueue("E", items);

Console.WriteLine($"front={front}, back={back}, count={count}");


The modulo operation wraps an index back to zero after it reaches the end of the array. This avoids shifting values after every dequeue.


```{index} deque
```

## Deques



A **deque**, pronounced like "deck", is a double-ended queue. It allows adding and removing at both the front and the back. That makes it more flexible than a stack or queue, but still more controlled than arbitrary list indexing.

C# does not provide a dedicated `Deque<T>` in the base class library. For small examples, `LinkedList<T>` can demonstrate double-ended operations.


In [ ]:
using System;
using System.Collections.Generic;

var deque = new LinkedList<string>();
deque.AddLast("middle");
deque.AddFirst("front");
deque.AddLast("back");

Console.WriteLine(deque.First!.Value);
Console.WriteLine(deque.Last!.Value);

deque.RemoveFirst();
deque.RemoveLast();

Console.WriteLine($"Remaining: {deque.First!.Value}");


A deque is useful when both ends have meaning, such as browser history, task stealing, or algorithms that need to expand from the front while sometimes adding urgent items to the front.


```{index} trace wraparound and distinguish full from empty
```

## Trace Wraparound and Distinguish Full from Empty



A ring buffer stores logical order across the end of physical storage. Start with capacity four, `front = back = count = 0`. Back is the next insertion slot, not the last occupied slot.

| Operation | Slots 0, 1, 2, 3 | Front | Back | Count | Logical order |
| --- | --- | ---: | ---: | ---: | --- |
| Enqueue A, B, C | A, B, C, unused | 0 | 3 | 3 | A, B, C |
| Dequeue | unused, B, C, unused | 1 | 3 | 2 | B, C |
| Enqueue D | unused, B, C, D | 1 | 0 | 3 | B, C, D |
| Enqueue E | E, B, C, D | 1 | 1 | 4 | B, C, D, E |
| Dequeue B | E, unused, C, D | 2 | 1 | 3 | C, D, E |

`front == back` occurs when the buffer is empty and when it is full. Count distinguishes those states. For logical position i, the physical slot is `(front + i) % Capacity`. The invariants are positive capacity, a front index inside the array, and `0 <= Count <= Capacity`. The next insertion slot is `(front + Count) % Capacity`.

**Trace:** Continue until empty, then enqueue F. Give every front, back, and count. Explain why physical slot zero does not always hold the oldest item.


```{index} a complete bounded generic queue
```

## A Complete Bounded Generic Queue



This implementation derives back from front and count, so it does not maintain a redundant third state variable. Enqueue checks for full storage before writing; dequeue checks emptiness before advancing front. Both clear or fill one slot and update count once.

Capacity is fixed. A full enqueue throws and preserves the waiting tickets. That overflow policy is part of the contract; silently overwriting the oldest ticket would implement different behavior. Capacity one is legal and exposes full/empty mistakes quickly.

**Test:** Fill a capacity-one queue, reject a second enqueue, remove the original item, reject an empty dequeue, and reuse it. Verify returned values and count after each request.


In [ ]:
using System;
var tickets = new RingQueue<string>(4);
tickets.Enqueue("A"); tickets.Enqueue("B"); tickets.Enqueue("C");
Console.WriteLine($"served={tickets.Dequeue()}");
tickets.Enqueue("D"); tickets.Enqueue("E");
Console.WriteLine($"count={tickets.Count}, front={tickets.Peek()}");
while (tickets.TryDequeue(out string ticket)) Console.WriteLine(ticket);
tickets.Enqueue("F"); Console.WriteLine($"reused={tickets.Dequeue()}");

public class RingQueue<T>
{
    private readonly T[] items;
    private int front;
    public int Count { get; private set; }
    public int Capacity => items.Length;
    public RingQueue(int capacity)
    {
        if (capacity <= 0) throw new ArgumentOutOfRangeException(nameof(capacity));
        items = new T[capacity];
    }
    public void Enqueue(T value)
    {
        if (Count == Capacity) throw new InvalidOperationException("Queue is full.");
        int back = (front + Count) % Capacity;
        items[back] = value;
        Count++;
    }
    public T Peek()
    {
        if (Count == 0) throw new InvalidOperationException("Queue is empty.");
        return items[front];
    }
    public T Dequeue()
    {
        T value = Peek();
        items[front] = default!;
        front = (front + 1) % Capacity;
        Count--;
        return value;
    }
    public bool TryDequeue(out T value)
    {
        if (Count == 0) { value = default!; return false; }
        value = Dequeue();
        return true;
    }
}

```{index} linked queue endpoints and growth choices
```

## Linked Queue Endpoints and Growth Choices



A singly linked queue uses a head for removal and a tail for insertion. Empty means count zero and both endpoints null. A nonempty chain reaches tail last, and `tail.Next` is null. Removing the only node must clear tail as well as head; otherwise a later enqueue may attach to a detached node.

Unlike removing the last item of a singly linked list, queue dequeue removes the head and needs no predecessor search. Each enqueue/dequeue performs a fixed number of link changes. The program still pays for allocation and memory management.

**Trace:** Draw both endpoints for enqueue A, dequeue A, and enqueue B. Identify the stale-tail bug if dequeue omits the empty case.


In [ ]:
using System;
var tickets = new LinkedQueue<string>();
tickets.Enqueue("A"); tickets.Enqueue("B");
Console.WriteLine(tickets.Dequeue()); Console.WriteLine(tickets.Dequeue());
tickets.Enqueue("C"); Console.WriteLine($"reused={tickets.Dequeue()}, count={tickets.Count}");

public class LinkedQueue<T>
{
    private class Node
    {
        public T Value;
        public Node? Next;
        public Node(T value) => Value = value;
    }
    private Node? head;
    private Node? tail;
    public int Count { get; private set; }
    public void Enqueue(T value)
    {
        var added = new Node(value);
        if (tail is null) head = added;
        else tail.Next = added;
        tail = added;
        Count++;
    }
    public T Peek()
    {
        if (head is null) throw new InvalidOperationException("Queue is empty.");
        return head.Value;
    }
    public T Dequeue()
    {
        T value = Peek();
        head = head!.Next;
        if (head is null) tail = null;
        Count--;
        return value;
    }
}

```{index} growing a ring and choosing a deque representation
```

## Growing a Ring and Choosing a Deque Representation



A growable ring allocates a larger array when full. Copy logical item i from `(front + i) % oldCapacity` into new slot i, then set front to zero. Count stays unchanged; the next insertion slot becomes count. For the full layout `[E, B, C, D]` with front one, growth must produce `[B, C, D, E, unused, ...]`, not `[E, B, C, D, ...]`.

The bounded ring above has O(1) worst-case operation work and no growth. A doubling ring has O(n) worst-case enqueue during resizing and O(1) amortized enqueue. `Queue<T>` provides a growable application collection; bounded buffers need an explicit reject, wait, or overwrite policy.

A deque can use a doubly linked list or a ring with both-end operations. In a ring, adding at the front moves front to `(front - 1 + Capacity) % Capacity`; removing the back uses `(front + Count - 1) % Capacity`. Validate full/empty states first. C# remainder can be negative, which is why the front formula adds capacity.

**Transfer:** Trace add-back A, add-front B, remove-back, remove-front at capacity three. Compare a ring's locality and spare slots with a doubly linked deque's node/link overhead.


```{index} queue; exercise
```

## Exercise



In [ ]:
// Exercise: Serve tickets
// Enqueue three ticket IDs: T-101, T-102, and T-103.
// Then dequeue and print each ticket in service order.

using System;
using System.Collections.Generic;

var tickets = new Queue<string>();

// Your code starts here.



// Your code ends here.


In [1]:
// Solution
using System;
using System.Collections.Generic;

var tickets = new Queue<string>();
tickets.Enqueue("T-101");
tickets.Enqueue("T-102");
tickets.Enqueue("T-103");

while (tickets.Count > 0)
{
    Console.WriteLine(tickets.Dequeue());
}


T-101
T-102
T-103


```{index} summary
```

## Summary



Queues model first-come, first-served order. Deques generalize the idea by allowing both ends to participate. Array-backed queues need ring-buffer logic; linked representations make end operations natural when both ends are tracked.


```{rubric} Footnotes
```

[^1]: The .NET base class library includes `Queue<T>` and `LinkedList<T>`, but not a dedicated general-purpose `Deque<T>` type.
